In [ ]:
import json
import time
import random
from datetime import datetime
import pytz
from azure.eventhub import EventHubProducerClient, EventData

# Load configuration
with open("config.json", "r", encoding="utf-8") as file:
    config = json.load(file)

# #Initialize Azure EventHub Client 
# producer = EventHubProducerClient.from_connection_string(
#     conn_str=config["azure_connection_string"],
#     eventhub_name=config["event_hub_name"]
# )

def generate_egyptian_plate():
    """Generate random Egyptian license plate."""
    letters = ['أ', 'ب','ت','ث','ج','ح','خ','د','ذ','ر','ز','س','ش','ص','ض','ط','ظ','ع','غ','ف','ق','ك','ل','م','ن','ه','و','ي']
    plate_letters = " ".join(random.choices(letters, k=3))
    plate_numbers = "".join(random.choices("123456789", k=4))
    return f"{plate_letters} {plate_numbers}"

def get_traffic_light():
    """Calculate traffic light state based on current time."""
    durations = config["traffic_light_durations"]
    total_cycle = durations["green"] + durations["red"] + durations["yellow"]
    cycle = int(time.time()) % total_cycle
    
    if cycle < durations["green"]: 
        return "Green"
    elif cycle < (durations["green"] + durations["red"]): 
        return "Red"
    else: 
        return "Yellow"

def generate_car_data():
    """Generate synthetic real-time data for a single vehicle."""
    cairo_tz = pytz.timezone('Africa/Cairo')
    current_time = datetime.now(cairo_tz)
    
    # Check time conditions
    is_weekend = current_time.weekday() in [4, 5]
    is_rush_hour = False if is_weekend else (16 <= current_time.hour < 20)
    
    traffic_light = get_traffic_light()
    lane = random.choice(config["lanes"])
    
    # Determine vehicle type (5% for emergency)
    is_emergency = random.random() < 0.05
    if is_emergency:
        v_type = random.choice(config["emergency_vehicles"])
    else:
        v_type = random.choice(config["vehicle_types"])

    # Truck weight simulation
    weight_tons = round(random.uniform(15.0, 60.0), 2) if v_type == "Heavy Truck" else 0.0

    # Event probabilities
    has_accident = random.random() < config["accident_probability"]
    pedestrians_crossing = random.random() < config["pedestrian_crossing_probability"]

    # Speed logic based on conditions
    if has_accident:
        speed = 0 
    elif is_emergency:
        speed = random.randint(90, 140)
    elif pedestrians_crossing:
        speed = random.randint(5, 15)
    else:
        if traffic_light == "Red":
            speed = random.randint(0, 5)
        elif traffic_light == "Yellow":
            speed = random.randint(10, 30)
        else: 
            speed = random.randint(*config["rush_hour_speed_range"]) if is_rush_hour else random.randint(*config["normal_speed_range"])
                
    # Inject anomalies (3%)
    if random.random() < config["anomaly_probability"] and not has_accident:
        speed = random.choice([-20, 250, 300])
        lane = random.choice([7, 8, 9])
        
    # Radar check
    ticket_caught = (speed > config["speed_limit"]) and not is_emergency

    return {
        "plate_number": generate_egyptian_plate(),
        "timestamp": current_time.isoformat(),
        "location": config["location"],
        "vehicle_type": v_type,
        "is_emergency": is_emergency,
        "weight_tons": weight_tons,
        "speed_kmh": speed,
        "lane": lane,
        "traffic_light": traffic_light,
        "is_weekend": is_weekend,
        "is_rush_hour": is_rush_hour,
        "accident_in_lane": has_accident,
        "pedestrians_crossing": pedestrians_crossing,
        "speeding_ticket_caught": ticket_caught
    }

def run_generator():
    """Main loop to simulate and send traffic stream."""
    try:
        # with producer:  
        while True:
            for _ in range(2):  # Generate 2 cars per second
                car_data = generate_car_data()
                json_data = json.dumps(car_data, ensure_ascii=False)
                
                ## Send to Azure 
                # event_data_batch = producer.create_batch()
                # event_data_batch.add(EventData(json_data))
                # producer.send_batch(event_data_batch)
                
                print(f" [SENT] {json_data}\n")
            
            time.sleep(1)
            
    except KeyboardInterrupt:
        print("\nStream stopped.")

if __name__ == "__main__":
    run_generator()